In [25]:
import os
import numpy as np
from csiread import Intel #Intel csı formatındaki .dat dosyalarını okumak için

In [26]:
file_path = "/Users/aleynagulkazdal/Desktop/Bitirme/WiAR-master/data/volunteer_a_all_data/csi_a10_12.dat"

csidata = Intel(file_path)
#.dat dosyasındaki CSI verileri belleğe yüklenir 
csidata.read()

#csidata.csi:
#Bu ham csı complex değerlerini içerir 
#Genelde shape: (T,30,3) olur
#T= zaman frame sayısı,30=subcarrier sayısı,3=anten sayısı
print("Rax CSI shape:",csidata.csi.shape)

Rax CSI shape: (194, 30, 3, 2)
connector_log=0x1
194 0xbb packets parsed


In [27]:
#2 = Real + Imag (kompleks bileşen)
#Real +Imag -> Complex sayıya çevireceğiz

#Real ve Imag bileşenlerini ayırıyoruz 
real_part = csidata.csi[...,0]
imag_part = csidata.csi[...,1]

#complex sayı oluştur
csi_complex = real_part + 1j * imag_part

#Amplitude (mutlak) al
csi_amp = np.abs(csi_complex)

print("Amplitude shape: ", csi_amp.shape)



Amplitude shape:  (194, 30, 3)


In [28]:
# (293, 30, 3) --> (293,90)

csi_flat = csi_amp.reshape(csi_amp.shape[0],-1)

print ("Flattened shape :",csi_flat.shape)


Flattened shape : (194, 90)


In [29]:
def fix_length(x, target=300):
    """
    x: (T, 90) shape'inde CSI amplitude verisi
    target: sabitlemek istediğimiz frame sayısı
    
    Eğer T > target:
        Fazla kısmı keseriz.
    Eğer T < target:
        Eksik kısmı sıfır padding ile doldururuz.
    """
    
    if x.shape[0] > target:
        # Fazla frame varsa kes
        return x[:target]
    
    elif x.shape[0] < target:
        # Eksik frame varsa sıfır ekle
        pad = np.zeros((target - x.shape[0], x.shape[1]))
        return np.vstack([x, pad])
    
    else:
        return x


x_fixed = fix_length(csi_flat)

print("Final shape:", x_fixed.shape)

Final shape: (300, 90)


In [30]:
data_root = "/Users/aleynagulkazdal/Desktop/Bitirme/WiAR-master/data"

volunteer_folders = [
    f for f in os.listdir(data_root)
    if f.startswith("volunteer") and not f.endswith(".rar")
    
]

print("Volunteer folders:",volunteer_folders)
print("Toplam volunteer:",len(volunteer_folders))


Volunteer folders: ['volunteerD_2', 'volunteer_b_all_data', 'volunteer_c_all_data', 'volunteer_9', 'volunteer_7', 'volunteerE_1', 'volunteer_8', 'volunteer_f_all_data', 'volunteerD_1', 'volunteer_a_all_data', 'volunteerE_2', 'volunteer_10']
Toplam volunteer: 12


In [31]:
#Kullanıcı isimlerini sadeleştirelim

def extract_user_id(folder_name):
    """
    volunteer_a_all_data --> a
    volunteerD_1 --> D
    """
    
    name = folder_name.replace("volunteer","")
    
    #başındaki "_" varsa kaldır
    if name.startswith("_"):
        name = name[1:]
        
    #Eğer _ varsa ilk kısmı al
    name = name.split("_")[0]    

    
    return name


all_user_ids = [extract_user_id(f) for f in volunteer_folders]
unique_user_ids = sorted(set(all_user_ids))

print("Unique Users:", unique_user_ids)
print("Toplam unique user:", len(unique_user_ids))


Unique Users: ['10', '7', '8', '9', 'D', 'E', 'a', 'b', 'c', 'f']
Toplam unique user: 10


In [32]:
#User --> integer label mapping

user_to_label = {user: idx for idx ,user in enumerate(unique_user_ids)}

print("User mapping:")
print(user_to_label)

User mapping:
{'10': 0, '7': 1, '8': 2, '9': 3, 'D': 4, 'E': 5, 'a': 6, 'b': 7, 'c': 8, 'f': 9}


In [33]:
#data set oluşturalım

X_list=[]
y_act_list = []
y_user_list = []


In [34]:
def process_csi_file(file_path, target=300):
    csidata = Intel(file_path)
    csidata.read()

    # Real + Imag → Complex
    real_part = csidata.csi[..., 0]
    imag_part = csidata.csi[..., 1]
    csi_complex = real_part + 1j * imag_part

    # Amplitude
    csi_amp = np.abs(csi_complex)

    # Flatten (30x3 → 90)
    csi_flat = csi_amp.reshape(csi_amp.shape[0], -1)

    # Fix length
    if csi_flat.shape[0] > target:
        return csi_flat[:target]
    elif csi_flat.shape[0] < target:
        pad = np.zeros((target - csi_flat.shape[0], csi_flat.shape[1]))
        return np.vstack([csi_flat, pad])
    else:
        return csi_flat

In [35]:
for folder in volunteer_folders:
    user_id = extract_user_id(folder)
    user_label = user_to_label[user_id]
    
    folder_path = os.path.join(data_root,folder)
    
    #O klasördeki tüm .dat dosyalarını al
    dat_files = [ f for f in os.listdir(folder_path) if f.endswith(".dat")]
    
    for file in dat_files:
        
        file_path = os.path.join(folder_path,file)
        
        #CSI işleme (FAZ 1'de yazdığımız fonk.)
        x = process_csi_file(file_path)
        
        #Activity label çıkar (csi_a6_21.dat -->6)
        act_str = file.split("_")[1] #a6
        act_label = int(act_str.replace("a","")) - 1 #0-based index
        
        X_list.append(x)
        y_act_list.append(act_label)
        y_user_list.append(user_label)

connector_log=0x1
201 0xbb packets parsed
connector_log=0x1
174 0xbb packets parsed
connector_log=0x1
201 0xbb packets parsed
connector_log=0x1
151 0xbb packets parsed
connector_log=0x1
190 0xbb packets parsed
connector_log=0x1
141 0xbb packets parsed
connector_log=0x1
183 0xbb packets parsed
connector_log=0x1
373 0xbb packets parsed
connector_log=0x1
167 0xbb packets parsed
connector_log=0x1
236 0xbb packets parsed
connector_log=0x1
284 0xbb packets parsed
connector_log=0x1
195 0xbb packets parsed
connector_log=0x1
200 0xbb packets parsed
connector_log=0x1
173 0xbb packets parsed
connector_log=0x1
177 0xbb packets parsed
connector_log=0x1
225 0xbb packets parsed
connector_log=0x1
175 0xbb packets parsed
connector_log=0x1
210 0xbb packets parsed
connector_log=0x1
165 0xbb packets parsed
connector_log=0x1
243 0xbb packets parsed
connector_log=0x1
172 0xbb packets parsed
connector_log=0x1
197 0xbb packets parsed
connector_log=0x1
190 0xbb packets parsed
connector_log=0x1
247 0xbb packets

In [36]:
X = np.array(X_list)
y_act= np.array(y_act_list)
y_user =np.array(y_user_list)

print("X shape:",X.shape)
print("y_act shape:",y_act.shape)
print("y_user shape:",y_user.shape)


X shape: (3521, 300, 90)
y_act shape: (3521,)
y_user shape: (3521,)


In [37]:
print("Activity dağılımı:")
print(np.bincount(y_act))

print("\nUser dağılımı:")
print(np.bincount(y_user))

Activity dağılımı:
[220 220 220 220 221 220 220 220 220 220 220 220 220 220 220 220]

User dağılımı:
[480 480 480 480   0   0 481 480 480 160]


In [38]:
# Önce gerçekten veri olan user'ları bul
from collections import Counter

user_counts = Counter(y_user)

valid_labels = [label for label, count in user_counts.items() if count > 0]

print("Valid user labels:", valid_labels)

Valid user labels: [np.int64(7), np.int64(8), np.int64(3), np.int64(1), np.int64(2), np.int64(9), np.int64(6), np.int64(0)]


In [39]:
mask = np.isin(y_user, valid_labels)

X = X[mask]
y_act = y_act[mask]
y_user = y_user[mask]

print("Filtre sonrası shape:", X.shape)
print("Filtre sonrası user dağılımı:", Counter(y_user))

Filtre sonrası shape: (3521, 300, 90)
Filtre sonrası user dağılımı: Counter({np.int64(6): 481, np.int64(7): 480, np.int64(8): 480, np.int64(3): 480, np.int64(1): 480, np.int64(2): 480, np.int64(0): 480, np.int64(9): 160})


In [40]:
unique_valid_users = sorted(set(y_user))

new_label_map = {old_label: new_idx 
                 for new_idx, old_label in enumerate(unique_valid_users)}

y_user = np.array([new_label_map[label] for label in y_user])

print("Yeni user dağılımı:", Counter(y_user))
print("Toplam user:", len(set(y_user)))

Yeni user dağılımı: Counter({np.int64(4): 481, np.int64(5): 480, np.int64(6): 480, np.int64(3): 480, np.int64(1): 480, np.int64(2): 480, np.int64(0): 480, np.int64(7): 160})
Toplam user: 8


In [41]:
np.save("X.npy", X)
np.save("y_act.npy", y_act)
np.save("y_user.npy", y_user)